# Latest Pokertool updates

This notebook demonstrates the shared chance-child interface and its connection to the exact turn, flop and preflop calculations. The central invariant is:

> Symmetry reduces repeated calculations, but every legal labelled child remains in the result.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from cards import hand_names, make_hand
from chance import (
    ChanceState,
    canonical_chance_children,
    canonical_state,
    chance_children,
    permute_state,
)
from flop import flop_outcomes
from showdown import INCOMPATIBLE
from turn import clear_showdown_cache, showdown_cache_info, turn_outcomes

## 1. One shared, non-recursive chance interface

`chance_children(state)` reveals only the next street. It does not evaluate descendants. Each child contains its revealed cards, next state and exact conditional probability.

In [2]:
first = make_hand(['Ac', 'Ad'])
second = make_hand(['Kc', 'Kd'])
preflop = ChanceState(first=first, second=second, action_history=('raise', 'call'))
flops = chance_children(preflop)
turns = chance_children(flops[0].state)
rivers = chance_children(turns[0].state)

pd.DataFrame([
    {'state': 'preflop', 'children': len(flops), 'each probability': str(flops[0].probability), 'next street': flops[0].state.street},
    {'state': 'flop', 'children': len(turns), 'each probability': str(turns[0].probability), 'next street': turns[0].state.street},
    {'state': 'turn', 'children': len(rivers), 'each probability': str(rivers[0].probability), 'next street': rivers[0].state.street},
])

,state,children,each probability,next street
0,preflop,17296,1/17296,flop
1,flop,45,1/45,turn
2,turn,44,1/44,river


In [3]:
print('Reveals:', [' '.join(hand_names(reveal)) for reveal in rivers[0].state.reveals])
print('Action history:', rivers[0].state.action_history)
print('River is terminal:', rivers[0].state.terminal)
print('Children after river:', chance_children(rivers[0].state))

Reveals: ['2c 3c 4c', '5c', '6c']
Action history: ('raise', 'call')
River is terminal: True
Children after river: ()


## 2. Public versus conditioned chance

A public turn leaves 48 candidate rivers. Once two compatible private hands are fixed, four additional cards are blocked and 44 remain.

In [4]:
turn_board = make_hand(['2h', '3h', '2s', '3s'])
public_rivers = chance_children(ChanceState(turn_board))
fixed_rivers = chance_children(ChanceState(turn_board, first, second))

print(f'Public turn:       {len(public_rivers)} rivers, probability {public_rivers[0].probability} each')
print(f'With fixed hands:  {len(fixed_rivers)} rivers, probability {fixed_rivers[0].probability} each')

Public turn:       48 rivers, probability 1/48 each
With fixed hands:  44 rivers, probability 1/44 each


## 3. Conditional suit symmetry

Children are grouped only under suit permutations that preserve the current board, both ordered private hands and reveal history. Every group stores its multiplicity, combined probability and transports from raw members to the representative.

In [5]:
turn_state = ChanceState(turn_board, first, second)
river_groups = canonical_chance_children(turn_state)
groups_table = pd.DataFrame([
    {
        'representative': ' '.join(hand_names(group.representative.reveal)),
        'multiplicity': group.multiplicity,
        'group probability': str(group.probability),
    }
    for group in river_groups
])
print(f'{len(fixed_rivers)} raw rivers -> {len(river_groups)} conditional suit orbits')
groups_table

44 raw rivers -> 22 conditional suit orbits


,representative,multiplicity,group probability
0,2c,2,1/22
1,3c,2,1/22
2,4c,2,1/22
3,5c,2,1/22
4,6c,2,1/22
5,7c,2,1/22
6,8c,2,1/22
7,9c,2,1/22
8,Tc,2,1/22
9,Jc,2,1/22


## 4. Turn calculation: all labels, fewer evaluations

`turn_outcomes` evaluates one representative per conditional orbit and expands the scalar showdown result back to every river label. A bounded canonical cache also reuses equivalent complete deals.

In [6]:
clear_showdown_cache()
turn_result = turn_outcomes(turn_board, first, second)
cache = showdown_cache_info()

print(f'Labelled rivers returned: {len(turn_result.rivers)}')
print(f'W/L/T: {turn_result.wins}/{turn_result.losses}/{turn_result.ties}')
print(f'Exact equity: {turn_result.equity} = {float(turn_result.equity):.2%}')
print(f'Canonical showdown cache misses: {cache.misses}')
print(f'Raw river count: {len(fixed_rivers)}')

Labelled rivers returned: 44
W/L/T: 42/2/0
Exact equity: 21/22 = 95.45%
Canonical showdown cache misses: 22
Raw river count: 44


In [7]:
pd.DataFrame([
    {'river': item.river_name, 'outcome': item.outcome}
    for item in turn_result.rivers
])

,river,outcome
0,2c,win
1,3c,win
2,4c,win
3,5c,win
4,6c,win
5,7c,win
6,8c,win
7,9c,win
8,Tc,win
9,Jc,win


## 5. Flop calculation

The same pattern is recursive: one representative turn is calculated per valid suit orbit, then transported back to all 45 labelled turns. Each transported turn result still contains all 44 labelled rivers.

In [8]:
flop_board = make_hand(['2h', '3h', '4h'])
flop_state = ChanceState(flop_board, first, second)
raw_turns = chance_children(flop_state)
turn_groups = canonical_chance_children(flop_state)
flop_result = flop_outcomes(flop_board, first, second)

print(f'Raw turns: {len(raw_turns)}')
print(f'Turn suit orbits calculated: {len(turn_groups)}')
print(f'Labelled turns returned: {len(flop_result.turns)}')
print(f'Ordered runouts represented: {flop_result.total:,}')
print(f'W/L/T: {flop_result.wins}/{flop_result.losses}/{flop_result.ties}')
print(f'Exact equity: {flop_result.equity} = {float(flop_result.equity):.2%}')
assert all(child.result.total == 44 for child in flop_result.turns)

Raw turns: 45
Turn suit orbits calculated: 34
Labelled turns returned: 45
Ordered runouts represented: 1,980
W/L/T: 1710/136/134
Exact equity: 1777/1980 = 89.75%


## 6. Preflop scale and symmetry

Two fixed hands leave 17,296 unordered flops. The complete exact result represents 34,246,080 ordered turn-river paths. Here we inspect the immediate flops and their symmetry groups without starting the full recursive evaluation.

In [9]:
preflop_children = chance_children(preflop)
preflop_groups = canonical_chance_children(preflop)

print(f'All labelled flops retained: {len(preflop_children):,}')
print(f'Conditional flop suit orbits: {len(preflop_groups):,}')
print(f'Ordered terminal paths represented: {len(preflop_children) * 45 * 44:,}')
print(f'Group probabilities sum to: {sum(group.probability for group in preflop_groups)}')

All labelled flops retained: 17,296
Conditional flop suit orbits: 5,502
Ordered terminal paths represented: 34,246,080
Group probabilities sum to: 1


Run `preflop_outcomes(first, second)` when the complete aggregate is required. It is deliberately omitted here because it recursively evaluates the full tree. `preflop_flop_outcomes(first, second)` provides the same labelled flop summaries incrementally.

## 7. Canonical keys and transport maps

`canonical_state` returns both a stable key and the exact suit permutation that moves all card-bearing components into canonical coordinates. Action history is preserved rather than merged.

In [10]:
key, transport = canonical_state(turn_state)
canonicalized = permute_state(turn_state, transport)

print('Transport:', transport)
print('Canonical board:', ' '.join(hand_names(canonicalized.board)))
print('Canonical first hand:', ' '.join(hand_names(canonicalized.first)))
print('Canonical second hand:', ' '.join(hand_names(canonicalized.second)))
print('Action history preserved:', canonicalized.action_history == turn_state.action_history)

Transport: (2, 3, 0, 1)
Canonical board: 2c 3c 2d 3d
Canonical first hand: Ah As
Canonical second hand: Kh Ks
Action history preserved: True


## Summary

- `chance_children` supplies complete one-street transitions and exact probabilities.
- `canonical_chance_children` supplies safe conditioned suit orbits, multiplicities and transports.
- Turn, flop and preflop outcome functions retain all labelled children.
- Representative calculations and the canonical showdown cache reduce repeated work.
- Fixed blockers and history are part of the state, so invalid symmetries are never used.